# Identifying Travel Personas using ACS PUMS 2024

AO, March 2026

**Background:** The TM team is estimating `UsualWorkSchoolLocation` (see [UsualWorkSchoolLocation: Decide how to handle remote workers](https://app.asana.com/1/11860278793487/project/13098083395690/task/1213196115898409?focus=true)).

Suzanne Childress penned the set of worker categories reprinted in the table below (see [Estimate residents/workers by permutations (inc wfh)](https://app.asana.com/1/11860278793487/project/13098083395690/task/1212672635539667?focus=true) - and the spreadsheet on [box](https://mtcdrive.app.box.com/file/2139481746457?s=drzys1bn5wp6y0s5p8rzvkyjsebz0rfq)). I added a sample identification column on how to capture, if at all, using the ACS PUMS data.
Several categories identified are not (local) TM relevant (e.g. retired Oregon residents [B5], Phoenix workers [B4]) - while others cannot be properly estimated given the questions in the ACS sample don't sufficiently get to only very sproradic behavior, such as one commute day per month, or the physical location of one's employer when working from home.

We populate the persona counts from ACS in this notebook.


| ID | Description | Example | Lives in Bay Area? | Sample identification |
| :--- | :--- | :--- | :--- | :--- |
| **A1** | Traditional commuter (internal) | Lives in Oakland, works in SF | Yes | STPUMA and POWSTPUMA in Bay Area FIPS (use w_bayarea and h_bayarea flags) |
| **A2** | Bay Area resident worker, mostly remote | Lives in Berkeley, employer in San Jose, goes in once a month | Yes | Like A1, but where JWTRNS ==11  |
| **A3** | Outbound commuter | Lives in Walnut Creek, works in Sacramento | Yes | Bay area resident, powpuma outside region|
| **A4** | Remote worker employed elsewhere | Lives in Palo Alto, company based in NYC | Yes | *cannot identify in PUMS sample as remote work precludes POW based powpuma*|
| **A5** | Not employed or not in the labor force | Retired, unemployed, student, caregiver | Yes | closest: COW is null OR SCHG in [15,16] |
| **A6** | Self-employed / home-based worker / fully-remote | Freelance designer in Alameda | Yes | COW in [6,7] and  JWTRNS ==11 |
| **B1** | Inbound commuter | Lives in Tracy, works in Livermore | No | POWPUMA in bay area (which we have code as w_bayarea)|
| **B2** | Occasional inbound commuter | Lives in Tahoe, goes to SF office once a month | No |  *cannot identify in PUMS sample as JWTRNS wouldn't capture once per month|
| **B3** | Nominal Bay Area job, fully remote | Lives in Austin, job officially based in SF | No | *cannot identify in out of area PUMS sample as remote work there precludes POW based powpuma connection to bay area*|
| **B4** | Irrelevant to Bay Area system | Lives and works in Phoenix | No | Skip as irrelevant. Any worker in US sample living and working elsewhere. |
| **B5** | Not employed, non-resident | Retired person in Oregon | No | Skip as irrelevant.|

# Preliminaries - imports, mappings, helper functions

In [73]:
import pandas as pd
import numpy as np
import pathlib
import os

import seaborn as sns
import matplotlib.pyplot as plt

# Path setup
M_DRIVE = pathlib.Path('/Volumes/Data/Models') if os.name=='posix' else pathlib.Path('M:/')
LODES_PATH = M_DRIVE / 'Data/Census/LEHD/RAW/lodes'
PUMS_PATH = M_DRIVE / 'Data/Census/PUMS'
CROSSWALK_PATH = M_DRIVE / 'Crosswalks/geo/block-to-puma'
OUT_PATH = M_DRIVE / 'Projects/Worker Flows/pums_to_lodes_reconciliation'
OUT_PATH.mkdir(parents=True, exist_ok=True)

# Analysis year
YEAR = 2024 
# Determine PUMA vintage based on PUMS year
# PUMS 2012-2021 use 2010-based PUMAs (vintage 2012)
# PUMS 2022+ use 2020-based PUMAs (vintage 2022)

PUMA_VINTAGE = '2024'
PUMA_LABEL = 'puma24'
POWPUMA_LABEL = 'POWSTPUMA_2024'
BLOCK_PUMA_XWALK_FILE = 'blocks2020_x_puma2024_xw.csv'
BLOCK_POWPUMA_XWALK_FILE = 'blocks2020_x_powpuma2024_xw.csv'
POWPUMA_TO_COUNTY_XWALK_FILE = 'powstpuma_to_county_2024.csv'
PUMA_TO_COUNTY_XWALK_FILE = 'stpuma_to_county_2024.csv'

print(f"Analysis Year: {YEAR}")
print(f"PUMA Vintage: {PUMA_VINTAGE} ({PUMA_LABEL})")
print(f"Block-to-PUMA Crosswalk: {BLOCK_PUMA_XWALK_FILE}")
print(f"PUMA-to-County Crosswalk: {PUMA_TO_COUNTY_XWALK_FILE}")
print(f"POWPUMA-to-County Crosswalk: {POWPUMA_TO_COUNTY_XWALK_FILE}")
print(f"LODES Blocks: 2020 vintage (LODES V8)")

Analysis Year: 2024
PUMA Vintage: 2024 (puma24)
Block-to-PUMA Crosswalk: blocks2020_x_puma2024_xw.csv
PUMA-to-County Crosswalk: stpuma_to_county_2024.csv
POWPUMA-to-County Crosswalk: powstpuma_to_county_2024.csv
LODES Blocks: 2020 vintage (LODES V8)


In [ ]:

# in case we want the SEs from standard errors - but pretty heavy for the national sample.

def calculate_replicate_se(primary_estimate, replicate_estimates, years=1):
    """
    Calculate standard error using replicate weights methodology.

    Args:
        primary_estimate: Estimate using primary weights
        replicate_estimates: Series of estimates using replicate weights
        years: Number of years the data represents (default: 1)

    Returns:
        float: Standard error
    """
    squared_diffs = (replicate_estimates - primary_estimate) ** 2
    variance = (4 / (80 * years)) * squared_diffs.sum()
    return np.sqrt(variance)


def calculate_confidence_interval(estimate, standard_error, confidence_level=0.90):
    """
    Calculate confidence interval and margin of error.

    Args:
        estimate: Point estimate
        standard_error: Standard error of the estimate
        confidence_level: Confidence level (default: 0.90 for 90% CI)

    Returns:
        dict with 'moe', 'ci_lower', 'ci_upper'
    """
    # note that Z-score for 90% confidence interval is 1.645 - 95% is 1.96
    z_score = 1.645 if confidence_level == 0.90 else 1.96
    
    moe = standard_error * z_score
    ci_upper = estimate + moe
    ci_lower = max(0, estimate - moe)  # Ensure non-negative

    return {
        'moe': moe,
        'ci_lower': ci_lower,
        'ci_upper': ci_upper
    }


def estimate_total_with_se(grouped_df, weight='PWGTP', years=1):
    """
    Calculate total estimate with standard errors using ACS PUMS replicate weights.

    This function calculates the total (sum) of a weighted variable along with
    associated standard errors, margins of error, and confidence intervals using
    the replicate weight methodology.

    Args:
        grouped_df: DataFrame or GroupBy object containing ACS PUMS data
        weight: Column name for primary weight (default: 'PWGTP')
        years: Number of years the data represents (default: 1)

    Returns:
        pd.Series with the following statistics:
            - 'total': Total estimate
            - 'se': Standard error
            - 'moe': Margin of error (90% CI)
            - 'ci_lower': Lower bound of 90% confidence interval
            - 'ci_upper': Upper bound of 90% confidence interval
            - 'cv': Coefficient of variation
            - 'sample_size': Number of records
    """
    # Identify replicate weight columns (PWGTP1, PWGTP2, ..., PWGTP80)
    repwgt_pattern = f'{weight}\\d{{1,2}}'

    # Calculate estimates (main weight, as well as replicates)
    primary_estimate = grouped_df[weight].sum() / float(years)
    replicate_estimates = grouped_df.filter(regex=repwgt_pattern).sum()

    # Calculate standard error
    se = calculate_replicate_se(primary_estimate, replicate_estimates, years)

    # Calculate confidence interval
    ci = calculate_confidence_interval(primary_estimate, se)

    # Coefficient of variation
    cv = se / primary_estimate if primary_estimate != 0 else np.inf

    return pd.Series({
        'total': int(primary_estimate),
        'se': se,
        'moe': ci['moe'],
        'ci_lower': np.ceil(ci['ci_lower']),
        'ci_upper': np.ceil(ci['ci_upper']),
        'cv': cv,
        'sample_size': grouped_df.shape[0]
    })



In [103]:

# Define Bay Area counties w fips 
bay_area_counties = {
    '06001': 'Alameda',
    '06013': 'Contra Costa',
    '06041': 'Marin',
    '06055': 'Napa',
    '06075': 'San Francisco',
    '06081': 'San Mateo',
    '06085': 'Santa Clara',
    '06095': 'Solano',
    '06097': 'Sonoma'
}

halofips = {
    '06077': 'San Joaquin',
    '06099': 'Stanislaus',
    '06067': 'Sacramento',
    '06047': 'Merced',
    '06113': 'Yolo',
    '06115': 'Yuba, Sutter',
    '06101': 'Yuba, Sutter',
    '06017': 'El Dorado',
    '06061': 'Placer',
    '06087': 'Santa Cruz',
    '06069': 'Monterey, San Benito',
    '06053': 'Monterey, San Benito',
    '06033': 'Mendocino, Lake',
    '06045': 'Mendocino, Lake',
}

cog_regions = {
    # ABAG/MTC
    '06001': 'ABAG/MTC',   # Alameda
    '06013': 'ABAG/MTC',   # Contra Costa
    '06041': 'ABAG/MTC',   # Marin
    '06055': 'ABAG/MTC',   # Napa
    '06075': 'ABAG/MTC',   # San Francisco
    '06081': 'ABAG/MTC',   # San Mateo
    '06085': 'ABAG/MTC',   # Santa Clara
    '06095': 'ABAG/MTC',   # Solano
    '06097': 'ABAG/MTC',   # Sonoma
    # SACOG
    '06017': 'SACOG',      # El Dorado
    '06061': 'SACOG',      # Placer
    '06067': 'SACOG',      # Sacramento
    '06101': 'SACOG',      # Sutter
    '06113': 'SACOG',      # Yolo
    '06115': 'SACOG',      # Yuba
    # SCAG
    '06025': 'SCAG',       # Imperial
    '06037': 'SCAG',       # Los Angeles
    '06059': 'SCAG',       # Orange
    '06065': 'SCAG',       # Riverside
    '06071': 'SCAG',       # San Bernardino
    '06111': 'SCAG',       # Ventura
    # Other COGs
    '06047': 'MCAG',       # Merced
    '06053': 'AMBAG',      # Monterey
    '06069': 'AMBAG',      # San Benito
    '06087': 'AMBAG',      # Santa Cruz
    '06077': 'SJCOG',      # San Joaquin
    '06099': 'STANCOG',    # Stanislaus
    '06073': 'SANDAG',     # San Diego    
    # Other
    '06033': 'Rest of CA', # Lake
    '06045': 'Mendocino CAG',
}
county_combo_list = list(bay_area_counties.values())+list(set(halofips.values()))

# Create reverse lookup (county name to FIPS)
county_fips = {v: k for k, v in bay_area_counties.items()}

## Recoding of crosswalks

In [104]:
# Load POWPUMA-to-county mapping 

puma_county_xwalk_path = CROSSWALK_PATH / PUMA_TO_COUNTY_XWALK_FILE
print("\nLoad PUMA-to-county mapping")
print(f"{puma_county_xwalk_path}")
puma_county_xwalk = pd.read_csv(
    puma_county_xwalk_path,dtype=str
)
# STPUMA concatenation uniquely identifies a puma
puma_county_xwalk = puma_county_xwalk.set_index("STPUMA")

# Load POWPUMA-to-county mapping 
powpuma_county_xwalk_path = CROSSWALK_PATH / POWPUMA_TO_COUNTY_XWALK_FILE
print("\nLoad POWPUMA-to-county mapping")
print(f"{powpuma_county_xwalk_path}")

powpuma_county_xwalk = pd.read_csv(
    powpuma_county_xwalk_path,dtype=str
)
powpuma_county_xwalk = powpuma_county_xwalk.set_index("POWSTPUMA")


Load PUMA-to-county mapping
M:\Crosswalks\geo\block-to-puma\stpuma_to_county_2024.csv

Load POWPUMA-to-county mapping
M:\Crosswalks\geo\block-to-puma\powstpuma_to_county_2024.csv


In [108]:
powpuma_county_xwalk['work_group'] = None
powpuma_county_xwalk.loc[powpuma_county_xwalk.fips.isin(bay_area_counties),'work_group']='Bay Area'
powpuma_county_xwalk.loc[powpuma_county_xwalk.fips.isin(halofips),'work_group']='Halo'


puma_county_xwalk['res_group'] = None
puma_county_xwalk.loc[puma_county_xwalk.fips.isin(bay_area_counties),'res_group']='Bay Area'
puma_county_xwalk.loc[puma_county_xwalk.fips.isin(halofips),'res_group']='Halo'


In [109]:
# add select cog grouping to crosswalk - we may have some PUMAS mapping to multiple counties in different COGs
puma_county_xwalk['ca_cog_region'] = (
    puma_county_xwalk.fips.str.split(', ')
    .explode()
    .map(cog_regions)
    .fillna('Outside CA Cogs')
    .groupby('STPUMA').apply(lambda x: '; '.join(set(x)))
)
puma_county_xwalk.query('counties=="San Mateo County"')



,Unnamed: 0,counties,fips,res_group,ca_cog_region
STPUMA,,,,,
0608101,352,San Mateo County,06081,Bay Area,ABAG/MTC
0608102,353,San Mateo County,06081,Bay Area,ABAG/MTC
0608103,354,San Mateo County,06081,Bay Area,ABAG/MTC
0608104,355,San Mateo County,06081,Bay Area,ABAG/MTC
0608105,356,San Mateo County,06081,Bay Area,ABAG/MTC
0608106,357,San Mateo County,06081,Bay Area,ABAG/MTC


In [110]:
# add select cog grouping to crosswalk - we may have some PUMAS mapping to multiple counties in different COGs
powpuma_county_xwalk['ca_cog_region'] = (
    powpuma_county_xwalk.fips.str.split(', ')
    .explode()
    .map(cog_regions)
    .fillna('Outside CA Cogs')
    .groupby('POWSTPUMA').apply(lambda x: '; '.join(set(x)))
)
powpuma_county_xwalk.query('counties=="San Francisco County"')

,Unnamed: 0,counties,fips,work_group,ca_cog_region
POWSTPUMA,,,,,
00607500,80,San Francisco County,06075,Bay Area,ABAG/MTC


## Prep PUMS national file load

In [131]:
# Load PUMS person files for the US as a whole so we can find IO commuters
# Note: PUMS data is split across multiple files per year (e.g. a/b split, psam_pusa.csv, psam_pusb.csv)
# Given large file, keep just minimal columns needed for OD flows and demographic stratification

# Column name aliases (vary across vintages)
COLUMN_ALIASES = {
    'STATE': ['ST', 'STATE'],
    'PUMA': ['PUMA', 'PUMA00', 'PUMA10'],
    "JWTR":["JWTR","JWTRNS"],
    'POWPUMA': ['POWPUMA', 'POWPUMA00', 'POWPUMA10'],
}

keep_cols = [
    'SERIALNO', 'PUMA', 'POWPUMA', 'POWSP', 'STATE','JWTR','COW','SCHG',
    'AGEP', 'WAGP', 'NAICSP', 'ESR', 'PWGTP'
]
weight_cols = [f'PWGTP{i}' for i in range(1,81)]

# Columns that must be read as strings to preserve leading zeros or non-numeric content
GEO_STR_COLS = {'SERIALNO', 'PUMA', 'POWPUMA', 'POWSP', 'STATE', 'NAICSP'}

def resolve_column_names(file_path, desired_cols):
    """Map desired column names to actual column names in file (handling vintage variations)."""
    # Read just the header
    header = pd.read_csv(file_path, nrows=0).columns.tolist()
    header_upper = {c.upper(): c for c in header}
    
    col_map = {}
    for desired in desired_cols:
        desired_upper = desired.upper()
        
        # Direct match
        if desired_upper in header_upper:
            col_map[desired] = header_upper[desired_upper]
        # Try aliases
        elif desired_upper in COLUMN_ALIASES:
            for alias in COLUMN_ALIASES[desired_upper]:
                if alias in header_upper:
                    col_map[desired] = header_upper[alias]
                    break
    
    return col_map

# Find PUMS files for the year
year_path = PUMS_PATH / f'PUMS {YEAR}'
if not year_path.exists():
    year_path = PUMS_PATH / str(YEAR)

pums_files = sorted(year_path.glob('psam_pus*.csv')) + sorted(year_path.glob(f'ss{str(YEAR)[-2:]}pus*.csv'))

if not pums_files:
    raise FileNotFoundError(f"No PUMS files found in {year_path}")

print(f"Found {len(pums_files)} PUMS file(s) for {YEAR}:")

# Load and concatenate all files
pums_parts = []
for pums_file in pums_files:
    col_map = resolve_column_names(pums_file, keep_cols + weight_cols)
    if not col_map:
        continue
    
    print(f"  Loading {pums_file.name}...")
    # Force str only for geo/identifier columns; let pandas infer numeric types for weights
    dtype_map = {v: str for k, v in col_map.items() if k in GEO_STR_COLS}
    df_part = pd.read_csv(
        pums_file, 
        usecols=list(col_map.values()),
        dtype=dtype_map
    )
    df_part = df_part.rename(columns={v: k for k, v in col_map.items()})
    pums_parts.append(df_part)

if not pums_parts:
    raise ValueError(f"No valid columns found in PUMS files for {YEAR}")

pums = pd.concat(pums_parts, ignore_index=True)
print(f"\nTotal records loaded: {len(pums):,}")
print(f"Replicate weight dtype: {pums['PWGTP1'].dtype}  (expect float64 or int64)")


Found 2 PUMS file(s) for 2024:
  Loading psam_pusa.csv...
  Loading psam_pusb.csv...

Total records loaded: 3,422,888
Replicate weight dtype: int64  (expect float64 or int64)


## PUMS recoding

In [132]:
# Convert numeric columns (handle both int and float formats)
pums['AGEP'] = pd.to_numeric(pums['AGEP'], errors='coerce')
pums['WAGP'] = pd.to_numeric(pums['WAGP'], errors='coerce')
pums['PWGTP'] = pd.to_numeric(pums['PWGTP'], errors='coerce')
pums['ESR'] = pd.to_numeric(pums['ESR'], errors='coerce')
pums['JWTR'] = pd.to_numeric(pums['JWTR'], errors='coerce')
pums['STATE2'] = pums['STATE'].str.zfill(2)
pums['PUMA5'] = pums['PUMA'].str.zfill(5)
pums['STPUMA'] = pums['STATE2'] + pums['PUMA5']

pums['STCOUNTY'] = pums['STPUMA'].map(puma_county_xwalk.fips)
pums['h_bayarea'] = pums['STCOUNTY'].isin(bay_area_counties)


# Filter to employed workers with valid workplace PUMA
# ESR: 1,2,4,5 = Civilian employed at work/with job, Armed forces at work/with job
pums_workers = pums[
    pums['ESR'].isin([1, 2, 4, 5])
    & pums['JWTR'].isin([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12]) # not 11 / work from home
].copy()

print(f"Employed workers with workplace: {len(pums_workers):,}")
print(f"Total weighted resident workers: {pums_workers['PWGTP'].sum():,.0f}")

Employed workers with workplace: 1,359,943
Total weighted resident workers: 143,402,419


In [160]:
# Format geographic codes properly
# Zero-pad and create composite codes
pums_workers['STATE2'] = pums_workers['STATE'].str.zfill(2)
pums_workers['PUMA5'] = pums_workers['PUMA'].str.zfill(5)
pums_workers['POWSP3'] = pums_workers['POWSP'].str.zfill(3)
pums_workers['POWPUMA5'] = pums_workers['POWPUMA'].str.zfill(5)

# Create full PUMA codes (State 2-digit + PUMA 5-digit = 7 chars)
pums_workers['STPUMA'] = pums_workers['STATE2'] + pums_workers['PUMA5']
pums_workers['POWSTPUMA'] = pums_workers['POWSP3'] + pums_workers['POWPUMA5']

# recode county to state

# Format PUMAs as STPUMA (state+puma, e.g., '0601301')
# User PUMAs are already formatted as state+puma
pums_workers['STCOUNTY'] = pums_workers['STPUMA'].map(puma_county_xwalk.fips)
pums_workers['h_counties'] = pums_workers['STPUMA'].map(puma_county_xwalk.counties)
pums_workers['h_counties_grp'] = pums_workers['STPUMA'].map(puma_county_xwalk.counties)
pums_workers['h_cog_region'] = pums_workers['STPUMA'].map(puma_county_xwalk.ca_cog_region).astype('category')
pums_workers['h_bayarea'] = pums_workers['STCOUNTY'].isin(bay_area_counties)
pums_workers['h_ca'] = pums_workers['STCOUNTY'].str.slice(0,2)=='06'

pums_workers['POWSTCOUNTY'] = pums_workers['POWSTPUMA'].map(powpuma_county_xwalk.fips)
pums_workers['w_counties'] = pums_workers['POWSTPUMA'].map(powpuma_county_xwalk.counties) 
pums_workers['w_cog_region'] = pums_workers['POWSTPUMA'].map(powpuma_county_xwalk.ca_cog_region).astype('category')
pums_workers['w_bayarea'] = pums_workers['POWSTCOUNTY'].isin(bay_area_counties)
pums_workers['w_ca'] = pums_workers['POWSTCOUNTY'].str.slice(0,2)=='06'
#pums_workers['POWSTCOUNTY'] = pums_workers['POWSTPUMA'].str.slice(1,6)


# Report mapping statistics
mapped_res = pums_workers['STCOUNTY'].notna().sum()
mapped_work = pums_workers['POWSTCOUNTY'].notna().sum()
print(f"  Mapped {mapped_res:,} / {len(pums_workers):,} residence PUMAs to counties ({100*mapped_res/len(pums_workers):.1f}%)")
print(f"  Mapped {mapped_work:,} / {len(pums_workers):,} workplace PUMAs to counties ({100*mapped_work/len(pums_workers):.1f}%)")
print(f"  Unique residence counties: {pums_workers['STCOUNTY'].nunique()}")
print(f"  Unique workplace counties: {pums_workers['POWSTCOUNTY'].nunique()}")

  Mapped 1,359,943 / 1,359,943 residence PUMAs to counties (100.0%)
  Mapped 1,359,273 / 1,359,943 workplace PUMAs to counties (100.0%)
  Unique residence counties: 1132
  Unique workplace counties: 1010


In [161]:
pums_workers['POWPUMA'].value_counts()

POWPUMA
00100    63710
03700    46964
03100    38294
00300    34755
00400    31088
         ...  
20300      341
07600      334
20400      324
00398      322
04798      309
Name: count, Length: 233, dtype: int64

In [138]:
# puma_county_xwalk['counties_grp'] = puma_county_xwalk.fips.replace(halofips)

# Analysis
## First a few quick tabulations


In [162]:
# employed residents, not working from home, by res county
pums_workers.query('h_bayarea').groupby('h_counties').apply(estimate_total_with_se)

C:\Users\aolsen\AppData\Local\Temp\ipykernel_30004\1877515973.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  pums_workers.query('h_bayarea').groupby('h_counties').apply(estimate_total_with_se)


,total,se,moe,ci_lower,ci_upper,cv,sample_size
h_counties,,,,,,,
Alameda County,"703,058","8,063","13,264","689,795","716,322",0,"6,815"
Contra Costa County,"458,665","6,485","10,668","447,998","469,333",0,"4,098"
Marin County,"93,909","2,848","4,684","89,225","98,594",0,835
Napa County,"55,100","1,709","2,812","52,288","57,913",0,627
San Francisco County,"367,417","5,058","8,320","359,097","375,738",0,"3,596"
San Mateo County,"325,307","4,518","7,432","317,876","332,739",0,"3,438"
Santa Clara County,"852,745","7,143","11,751","840,995","864,496",0,"8,711"
Solano County,"182,814","3,230","5,314","177,501","188,128",0,"1,704"
Sonoma County,"206,112","3,239","5,328","200,784","211,441",0,"2,042"


In [163]:
# employed residents, not working from home, by cog region
pd.options.display.float_format = '{:,.0f}'.format
pums_workers.groupby(['h_cog_region']).apply(estimate_total_with_se)

C:\Users\aolsen\AppData\Local\Temp\ipykernel_30004\2180464462.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  pums_workers.groupby(['h_cog_region']).apply(estimate_total_with_se)
C:\Users\aolsen\AppData\Local\Temp\ipykernel_30004\2180464462.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  pums_workers.groupby(['h_cog_region']).apply(estimate_total_with_se)


,total,se,moe,ci_lower,ci_upper,cv,sample_size
h_cog_region,,,,,,,
ABAG/MTC,"3,245,127","17,606","28,962","3,216,165","3,274,090",0,"31,866"
AMBAG,"313,875","4,450","7,320","306,555","321,196",0,"3,190"
MCAG,"116,124","2,696","4,436","111,689","120,560",0,983
Outside CA Cogs,"128,881,698","107,182","176,315","128,705,384","129,058,013",0,"1,221,367"
Rest of CA; Mendocino CAG,"54,109","2,042","3,360","50,750","57,469",0,498
SACOG,"1,035,539","8,882","14,610","1,020,929","1,050,150",0,"9,782"
SANDAG,"1,407,568","9,792","16,108","1,391,460","1,423,677",0,"13,718"
SCAG,"7,815,740","24,576","40,428","7,775,313","7,856,168",0,"73,756"
SJCOG,"315,042","4,787","7,875","307,167","322,918",0,"2,798"


In [164]:
# employed residents, not working from home, by work cog region
pums_workers.groupby(['w_cog_region']).apply(estimate_total_with_se)

C:\Users\aolsen\AppData\Local\Temp\ipykernel_30004\884744011.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  pums_workers.groupby(['w_cog_region']).apply(estimate_total_with_se)
C:\Users\aolsen\AppData\Local\Temp\ipykernel_30004\884744011.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  pums_workers.groupby(['w_cog_region']).apply(estimate_total_with_se)


,total,se,moe,ci_lower,ci_upper,cv,sample_size
w_cog_region,,,,,,,
ABAG/MTC,"3,420,363","18,233","29,994","3,390,370","3,450,357",0,"33,471"
AMBAG,"286,829","5,521","9,083","277,747","295,912",0,"2,889"
MCAG,"89,306","2,939","4,835","84,472","94,141",0,799
Outside CA Cogs,"128,780,524","106,994","176,005","128,604,520","128,956,529",0,"1,220,442"
Rest of CA; Mendocino CAG,"48,416","2,046","3,366","45,050","51,783",0,441
SACOG,"1,023,335","9,163","15,073","1,008,263","1,038,408",0,"9,648"
SANDAG,"1,445,819","10,795","17,757","1,428,062","1,463,577",0,"13,960"
SCAG,"7,766,942","24,581","40,436","7,726,506","7,807,379",0,"73,379"
SJCOG,"281,265","5,725","9,417","271,848","290,683",0,"2,507"


In [165]:
# how many employed residents not at work?
pums.query('ESR==2 & h_bayarea').groupby('STCOUNTY').PWGTP.sum()

STCOUNTY
06001    20510
06013    15020
06041     3671
06055     1177
06075    12347
06081    10110
06085    30475
06095     6658
06097     5191
Name: PWGTP, dtype: int64

In [166]:
# non-bay area workers and residents
pums_workers.query('(~h_bayarea & ~w_bayarea)').groupby('STATE').PWGTP.sum().sum()

np.int64(139930932)

## Persona identification

In [167]:

# Travel Persona Classification Implementation

p = pums.copy()

# Numeric recodings
p['ESR_num']  = pd.to_numeric(p['ESR'],  errors='coerce')
p['JWTR_num'] = pd.to_numeric(p['JWTR'], errors='coerce')
p['PWGTP']    = pd.to_numeric(p['PWGTP'],errors='coerce')
p['COW_num']  = pd.to_numeric(p['COW'],  errors='coerce')
p['SCHG_num'] = pd.to_numeric(p['SCHG'], errors='coerce')

# Geographic codes
p['STATE2']   = p['STATE'].str.zfill(2)
p['PUMA5']    = p['PUMA'].str.zfill(5)
p['POWSP3']   = p['POWSP'].str.zfill(3)
p['POWPUMA5'] = p['POWPUMA'].str.zfill(5)
p['STPUMA']    = p['STATE2'] + p['PUMA5']
p['POWSTPUMA'] = p['POWSP3'] + p['POWPUMA5']

p['STCOUNTY']    = p['STPUMA'].map(puma_county_xwalk.fips)
p['POWSTCOUNTY'] = p['POWSTPUMA'].map(powpuma_county_xwalk.fips)
p['h_bayarea']   = p['STCOUNTY'].isin(bay_area_counties)
p['w_bayarea']   = p['POWSTCOUNTY'].isin(bay_area_counties)

# Create boolean series for easy selection
employed  = p['ESR_num'].isin([1, 2, 4, 5])
remote    = p['JWTR_num'] == 11
commutes  = p['JWTR_num'].between(1, 10) | (p['JWTR_num'] == 12)
# COW: 6 = Self-employed, not incorporated; 7 = Self-employed, incorporated
self_emp  = p['COW_num'].isin([6, 7])
# SCHG: 15 = College undergraduate; 16 = Graduate/professional school
student   = p['SCHG_num'].isin([15, 16])

# The deal is that there is some universe overlap here - so we set a hierarchy / prioritization order
# with assignment -- most specific first
conditions = [
    p['h_bayarea'] & employed & remote & self_emp,           # A6
    p['h_bayarea'] & p['w_bayarea'] & employed & remote & ~self_emp,  # A2
    p['h_bayarea'] & p['w_bayarea'] & employed & commutes,   # A1
    p['h_bayarea'] & ~p['w_bayarea'] & employed,             # A3
    ~p['h_bayarea'] & p['w_bayarea'] & employed,             # B1
    p['h_bayarea'] & student,                                # A5s - disambiguation of a5
    p['h_bayarea'] & ~employed,                              # A5x - disambiguation of a5
    ~p['h_bayarea'] & ~p['w_bayarea'],                       # B4/B5
]
choices = ['A6', 'A2', 'A1', 'A3', 'B1', 'A5s', 'A5x', 'B4/B5']

# Very efficient handling of multiple conditions in one go - and order matters.
# Careful: Watch out for consistency between conditions and choices list!
p['persona'] = np.select(conditions, choices, default='Unclassified')

print("=== Mutual Exclusivity Check ===")
print(f"Total records:  {len(p):,}")
print(f"Unclassified:   {(p['persona']=='Unclassified').sum():,}")
print()


=== Mutual Exclusivity Check ===
Total records:  3,422,888
Unclassified:   0



## Persona summary

In [170]:

persona_labels = {
    'A1':    'Traditional internal commuter',
    'A2':    'Bay Area resident, regular employee WFH',
    'A3':    'Outbound commuter (Bay Area to outside)',
    'A6':    'Self-employed / home-based Bay Area worker',
    'B1':    'Inbound commuter (outside to Bay Area)',
    'A5s':   'Bay Area college/grad student',
    'A5x':   'Not employed / not in labor force (Bay Area, non-student)',
    'B4/B5': 'Not Bay Area relevant (skip)',
    'Unclassified': 'Unclassified',
}

summary = (
    p.groupby('persona')
     .agg(
         description  = ('persona', lambda x: persona_labels.get(x.iloc[0], '')),
         records      = ('PWGTP',   'count'),
         weighted_pop = ('PWGTP',   'sum'),
     )
     .reindex(['A1','A2','A6','A3','B1','A5s','A5x','B4/B5','Unclassified'])
     .dropna(how='all')
)

# Denominators are a little weird here:
#   A-series employed (A1,A2,A6,A3): Bay Area employed workers (h_bayarea & employed)
#   A-series non-employed (A5s,A5x): Bay Area non-employed residents (h_bayarea & ~employed)
#   B1:    all workers with a Bay Area workplace (w_bayarea & employed)
#   B4/B5: no percentage (irrelevant universe)
denom_h_emp     = p.loc[p['h_bayarea'] & employed,  'PWGTP'].sum()
denom_h_nonemp  = p.loc[p['h_bayarea'] & ~employed, 'PWGTP'].sum()
denom_bay_workers = p.loc[p['w_bayarea'] & employed, 'PWGTP'].sum()

emp_series    = ['A1', 'A2', 'A6', 'A3']
nonemp_series = ['A5s', 'A5x']
summary['pct'] = np.nan
summary.loc[emp_series,    'pct'] = summary.loc[emp_series,    'weighted_pop'] / denom_h_emp    * 100
summary.loc[nonemp_series, 'pct'] = summary.loc[nonemp_series, 'weighted_pop'] / denom_h_nonemp * 100
summary.loc['B1',          'pct'] = summary.loc['B1',          'weighted_pop'] / denom_bay_workers * 100

summary['denominator'] = ''
summary.loc[emp_series,    'denominator'] = f'% of Bay Area employed workers ({denom_h_emp/1e6:.2f}M)'
summary.loc[nonemp_series, 'denominator'] = f'% of Bay Area non-employed residents ({denom_h_nonemp/1e6:.2f}M)'
summary.loc['B1',          'denominator'] = f'% of Bay Area-bound workers ({denom_bay_workers/1e6:.2f}M)'

print(f"Bay Area employed workers (h_bayarea & employed):      {denom_h_emp:>12,.0f}")
print(f"Bay Area non-employed residents (h_bayarea & ~employed):{denom_h_nonemp:>11,.0f}")
print(f"All workers with Bay Area workplace (w_bayarea):       {denom_bay_workers:>12,.0f}")
print()
print("=== Persona Summary ===")
display(
    summary[['description','records','weighted_pop','pct','denominator']].style.format({
        'records':      '{:,.0f}',
        'weighted_pop': '{:,.0f}',
        'pct':          lambda v: f'{v:.1f}%' if pd.notna(v) else '—',
    })
)


Bay Area employed workers (h_bayarea & employed):         4,014,351
Bay Area non-employed residents (h_bayarea & ~employed):  3,634,016
All workers with Bay Area workplace (w_bayarea):          4,084,422

=== Persona Summary ===


,description,records,weighted_pop,pct,denominator
persona,,,,,
A1,Traditional internal commuter,"31,334","3,194,003",79.6%,% of Bay Area employed workers (4.01M)
A2,"Bay Area resident, regular employee WFH","5,821","552,132",13.8%,% of Bay Area employed workers (4.01M)
A6,Self-employed / home-based Bay Area worker,"1,263","111,927",2.8%,% of Bay Area employed workers (4.01M)
A3,Outbound commuter (Bay Area to outside),"1,468","156,289",3.9%,% of Bay Area employed workers (4.01M)
B1,Inbound commuter (outside to Bay Area),"2,137","226,360",5.5%,% of Bay Area-bound workers (4.08M)
A5s,Bay Area college/grad student,"2,392","235,366",6.5%,% of Bay Area non-employed residents (3.63M)
A5x,"Not employed / not in labor force (Bay Area, non-student)","36,273","3,398,650",93.5%,% of Bay Area non-employed residents (3.63M)
B4/B5,Not Bay Area relevant (skip),"3,342,200","332,236,263",—,


## Table Classification Revisited

> We populate the initial table, using the national ACS PUMS files for 2024.  Percentages use three context-appropriate denominators:
> - A-series employed (A1, A2, A6, A3): denominator = Bay Area employed workers = 4,014,351
> - A-series not employed / not in lf (A5s, A5x): denominator = Bay Area non-employed residents = 3,634,016
> - B1 (inbound commuters): denominator = all workers with a Bay Area workplace (inbound + internal) = 4,084,422
> - B4/B5: no percentage — Largely TM-irrelevant universe

| ID | Description | Identifiable? | Sample records | Weighted persons | % | Denominator | Identification criteria |
| :--- | :--- | :---: | ---: | ---: | ---: | :--- | :--- |
| **A1** | Traditional internal commuter | Yes | 31,334 | 3,194,003 | 79.6% | of Bay Area employed workers | `h_bayarea & w_bayarea & ESR in {1,2,4,5} & JWTR not 11` |
| **A2** | Bay Area resident, regular employee WFH | Yes | 5,821 | 552,132 | 13.8% | of Bay Area employed workers | `h_bayarea & w_bayarea & ESR in {1,2,4,5} & JWTR==11 & COW not in {6,7}` |
| **A6** | Self-employed / home-based Bay Area worker | Yes | 1,263 | 111,927 | 2.8% | of Bay Area employed workers | `h_bayarea & ESR in {1,2,4,5} & JWTR==11 & COW in {6,7}` |
| **A3** | Outbound commuter | Yes | 1,468 | 156,289 | 3.9% | of Bay Area employed workers | `h_bayarea & w_bayarea==False & ESR in {1,2,4,5}` |
| **A4** | Remote worker, employer outside region | No | — | — | — | — | POW recoded to home for remote workers; employer unknowable |
| **A5s** | Bay Area college / grad student | Yes | 2,392 | 235,366 | 6.5% | of Bay Area non-employed residents | `h_bayarea & SCHG in {15,16}` |
| **A5x** | Not employed / not in labor force (non-student) | Yes | 36,273 | 3,398,650 | 93.5% | of Bay Area non-employed residents | `h_bayarea & ESR not in {1,2,4,5} & SCHG not in {15,16}` |
| **B1** | Inbound commuter | Yes | 2,137 | 226,360 | 5.5% | of Bay Area-bound workers | `h_bayarea==False & w_bayarea & ESR in {1,2,4,5}` |
| **B2** | Occasional inbound commuter | No | — | — | — | — | JWTR reflects usual mode; once-a-month trips not captured |
| **B3** | Nominal Bay Area job, fully remote (out-of-state) | No | — | — | — | — | Out-of-state PUMS has POW recoded to home |
| **B4/B5** | Not Bay Area relevant | Skipped | — | — | — | — | Excluded by design |


### Key observations
- Among Bay Area employed residents (4.0M), 80% are traditional commuters (A1), 14% are remote employees (A2), 4% are outbound commuters (A3), and 3% are self-employed remote workers (A6). A1+A2+A6+A3 sum to ~100% of the employed base by construction.
- Among Bay Area *non-employed* residents (3.6M), 93.5% are not in the labor force (A5x) and 6.5% are college/grad students (A5s).
- Inbound commuters (B1) represent 5.5% of all Bay Area-bound employment, with the remaining 94.5% being Bay Area residents commuting internally (A1 + A2 + A6).
- A6 (112K, 2.8% of employed) is meaningfully distinct from A2 — self-employed remote workers have different trip-making propensity than regular employees working from home.
- Three personas remain structurally unobservable: A4, B2, and B3 — all masked by the Census Bureau's practice of recoding remote workers' POW to their home address.

Fully remote workers (A4 and A6) can be obtained from the **SWAA survey** - though without specificity to the Bay Area. We also note that A4 and A6 differ mainly in a class of worker sense - one being self employed, the other being a wage / salary employee (for an out of region company.)
